# Artoria Neural Style Transfer
## A Kaggle-Ready LoRA Training Notebook

> **Project context**  
> This work was developed for the **Machine Learning Fundamentals** course in the **Artificial Intelligence and Data Engineering** department. The project topic was selected by me and my teammates, and the dataset was built by our team.

This notebook reconstructs the full training pipeline needed to publish the project properly on Kaggle and GitHub. It is designed to train four reusable LoRA adapters for artistic style transfer while respecting Kaggle's compute, storage, and package constraints.

## Notebook Goals

| Goal | Output |
|---|---|
| Rebuild the missing training pipeline | A public, reusable Kaggle notebook |
| Train four style-specific LoRAs | `cubism`, `pop-art`, `post-impressionism`, `ukiyo-e` |
| Stay stable on Kaggle | Single-T4 by default, optional 2xT4 parallel mode |
| Keep artifacts publication-ready | Weights, logs, validation grid, run summary, release zip |

> **Recommended first run**  
> Keep the notebook in `single_gpu` mode for the first successful training pass. Switch to `dual_gpu_parallel` only after the environment is confirmed stable.

## Pipeline Architecture

```text
Kaggle Dataset Input
    |
    +--> CCubism
    +--> CPop_Art
    +--> CPost_Impressionism
    +--> CUkiyo_e
           |
           v
Lightweight Workspace Builder
  - validates folder structure
  - samples up to N images per style
  - prefers symlinks, falls back to copies
  - writes metadata.jsonl captions
           |
           v
Stable Diffusion 1.5 + LoRA Training
  - fp16
  - 8-bit Adam (with AdamW fallback)
  - gradient checkpointing
  - sequential by default
  - optional 2xT4 parallel launch
           |
           v
Artifact Packager
  - safetensors weights
  - logs
  - validation grid
  - JSON run summary
  - release zip
```

## Kaggle Constraints and Design Choices

1. Kaggle sessions are time-bound (~9h on T4), so the notebook uses a lightweight LoRA setup instead of full fine-tuning.
2. Kaggle output storage is limited, so Hugging Face caches are redirected to `/tmp` instead of `/kaggle/working`.
3. Broad package upgrades often break the preinstalled CUDA stack, so the bootstrap cell only installs a narrow pinned set and never touches `torch`.
4. `--report_to=tensorboard` is used (not `none`) because newer versions of `accelerate` dropped support for the `none` logger.
5. Dual T4 support is implemented as **two independent single-GPU training jobs** rather than a more fragile distributed setup.

## Step 1 — Environment Audit

In [ ]:
import importlib.metadata as _meta
import os
import platform
import sys
from pathlib import Path

import torch

print(f'Python: {platform.python_version()}')
print(f'Platform: {platform.platform()}')
print(f'Torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for index in range(torch.cuda.device_count()):
    print(f'  GPU {index}: {torch.cuda.get_device_name(index)}')
    free_mem = torch.cuda.mem_get_info(index)[0] / (1024 ** 3)
    total_mem = torch.cuda.mem_get_info(index)[1] / (1024 ** 3)
    print(f'  GPU {index} memory: {free_mem:.1f} GB free / {total_mem:.1f} GB total')

for pkg in ['diffusers', 'transformers', 'accelerate', 'peft', 'bitsandbytes', 'huggingface_hub']:
    try:
        print(f'{pkg}: {_meta.version(pkg)}')
    except _meta.PackageNotFoundError:
        print(f'{pkg}: NOT INSTALLED')

## Step 2 — Environment Bootstrap

This cell installs only the packages needed for LoRA training and never touches `torch` or CUDA.

> **If the bootstrap installs or upgrades packages, the kernel will restart automatically. Continue from Step 3.**

In [ ]:
import importlib.metadata as _meta
import subprocess
import sys
from packaging.version import Version

# Pinned ranges that work on Kaggle's Python 3.12 / CUDA 12.x image.
# accelerate>=0.31 dropped 'none' from LoggerType — use tensorboard instead.
target_packages = {
    'accelerate':      ('0.33.0', '2.0.0'),
    'bitsandbytes':    ('0.43.0', '1.0.0'),
    'diffusers':       ('0.27.0', '1.0.0'),
    'ftfy':            ('6.0.0',  '8.0.0'),
    'peft':            ('0.10.0', '1.0.0'),
    'sentencepiece':   ('0.1.99', '1.0.0'),
    'transformers':    ('4.40.0', '5.0.0'),
}

def _in_range(ver_str, lo, hi):
    v = Version(ver_str)
    return Version(lo) <= v < Version(hi)

to_install = []
for pkg, (lo, hi) in target_packages.items():
    try:
        installed = _meta.version(pkg)
        if not _in_range(installed, lo, hi):
            print(f'  {pkg}: {installed} is outside [{lo}, {hi}) — will upgrade')
            to_install.append(f'{pkg}>={lo},<{hi}')
        else:
            print(f'  {pkg}: {installed} OK')
    except _meta.PackageNotFoundError:
        print(f'  {pkg}: missing — will install')
        to_install.append(f'{pkg}>={lo},<{hi}')

if to_install:
    print('\nInstalling:', ', '.join(to_install))
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--no-cache-dir', '--upgrade', *to_install
    ])
    print('\nBootstrap complete. Restarting kernel...')
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip and getattr(ip, 'kernel', None):
            ip.kernel.do_shutdown(restart=True)
    except Exception:
        pass
    print('Please restart the kernel manually and continue from Step 3.')
else:
    print('\nAll packages already satisfy version requirements.')

## Step 3 — Training Configuration

Dataset paths match the Kaggle dataset layout. `max_images_per_style=500` balances quality with T4 time limits.  
Styles with fewer images automatically use all available.  
Training steps are sized so all four styles complete within ~6–7h on a single T4.

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    project_name:               str
    base_model:                 str
    train_mode:                 str     # 'single_gpu' | 'dual_gpu_parallel'
    max_images_per_style:       int
    content_images_path:        Path
    max_train_steps:            int
    learning_rate:              float
    rank:                       int
    resolution:                 int
    train_batch_size:           int
    gradient_accumulation_steps: int
    seed:                       int
    workspace_root:             Path
    output_root:                Path
    temp_root:                  Path
    use_8bit_adam:              bool    # False = safer AdamW fallback

# ── Adjust these dataset paths to match your Kaggle dataset mount ──────────
# The default below assumes the dataset is mounted at:
#   /kaggle/input/artoria-dataset/
# If your mount point is different, update DATASET_ROOT accordingly.
DATASET_ROOT = Path('/kaggle/input/artoria-dataset/CleanedDataSet')

STYLE_SPECS = [
    {
        'style_id':     'cubism',
        'display_name': 'Cubism',
        'dataset_path': str(DATASET_ROOT / 'CCubism'),
        'caption':      'an artwork in the cubism style',
        'prompt_suffix': 'cubism style, geometric abstraction, angular composition',
    },
    {
        'style_id':     'pop-art',
        'display_name': 'Pop Art',
        'dataset_path': str(DATASET_ROOT / 'CPop_Art'),
        'caption':      'an artwork in the pop art style',
        'prompt_suffix': 'pop art style, bold color fields, graphic contrast',
    },
    {
        'style_id':     'post-impressionism',
        'display_name': 'Post-Impressionism',
        'dataset_path': str(DATASET_ROOT / 'CPost_Impressionism'),
        'caption':      'an artwork in the post-impressionism style',
        'prompt_suffix': 'post-impressionism style, expressive brushwork, vivid color rhythm',
    },
    {
        'style_id':     'ukiyo-e',
        'display_name': 'Ukiyo-e',
        'dataset_path': str(DATASET_ROOT / 'CUkiyo_e'),
        'caption':      'an artwork in the ukiyo-e style',
        'prompt_suffix': 'ukiyo-e style, woodblock print aesthetics, elegant contour lines',
    },
]

CONFIG = TrainingConfig(
    project_name             = 'artoria-style-transfer',
    base_model               = 'runwayml/stable-diffusion-v1-5',
    train_mode               = 'single_gpu',   # change to 'dual_gpu_parallel' if 2xT4
    max_images_per_style     = 500,            # 500 images × 4 styles = manageable T4 run
    content_images_path      = DATASET_ROOT / 'Content',
    max_train_steps          = 600,            # ~45 min per style on T4
    learning_rate            = 1e-4,
    rank                     = 8,              # reduced rank → faster + less VRAM
    resolution               = 512,
    train_batch_size         = 1,
    gradient_accumulation_steps = 4,
    seed                     = 42,
    workspace_root           = Path('/kaggle/working/style_transfer_workspace'),
    output_root              = Path('/kaggle/working/artoria_release'),
    temp_root                = Path('/tmp/artoria_style_transfer'),
    use_8bit_adam            = True,           # set False if bitsandbytes fails
)

import pprint
pprint.pprint(asdict(CONFIG))

## Step 4 — Dataset Validation

In [ ]:
from pathlib import Path
import pandas as pd

VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.tif'}

def count_images(directory: Path) -> int:
    if not directory.exists():
        return 0
    return sum(1 for p in directory.rglob('*')
               if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS)

rows = []
for spec in STYLE_SPECS:
    source_dir = Path(spec['dataset_path'])
    rows.append({
        'style_id':    spec['style_id'],
        'display_name': spec['display_name'],
        'dataset_path': str(source_dir),
        'exists':      source_dir.exists(),
        'image_count': count_images(source_dir),
    })

content_dir = CONFIG.content_images_path
rows.append({
    'style_id':    'content',
    'display_name': 'Content (real-life objects)',
    'dataset_path': str(content_dir),
    'exists':      content_dir.exists(),
    'image_count': count_images(content_dir),
})

dataset_report = pd.DataFrame(rows)
display(dataset_report)

missing = dataset_report.loc[~dataset_report['exists'], 'dataset_path'].tolist()
if missing:
    print('\n⚠️  MISSING paths — update DATASET_ROOT in Step 3:')
    for p in missing:
        print(f'   {p}')
    # List what is actually available under /kaggle/input/
    print('\nAvailable under /kaggle/input/:')
    for p in sorted(Path('/kaggle/input').rglob('*')):
        if p.is_dir():
            print(f'  {p}')
    raise FileNotFoundError(f'Missing dataset paths: {missing}')
else:
    print('✅ All dataset paths found.')

## Step 5 — Workspace Preparation

Builds a writable workspace in `/kaggle/working` (Kaggle input dirs are read-only).  
Uses symlinks first; falls back to file copies when symlinks are unavailable.

In [ ]:
import json
import os
import random
import shutil
from pathlib import Path

import pandas as pd

def prepare_style_workspace(config, style_specs):
    rng = random.Random(config.seed)
    ws = config.workspace_root
    if ws.exists():
        shutil.rmtree(ws)
    ws.mkdir(parents=True, exist_ok=True)

    prepared = []
    total_images = 0
    total_copy_bytes = 0

    for spec in style_specs:
        source_dir = Path(spec['dataset_path'])
        image_paths = sorted(
            p for p in source_dir.rglob('*')
            if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
        )
        if not image_paths:
            raise RuntimeError(f'No images found in {source_dir}')

        n_avail = len(image_paths)
        n_limit = config.max_images_per_style
        if n_limit > 0 and n_avail > n_limit:
            selected = sorted(rng.sample(image_paths, n_limit))
            note = f'sampled {n_limit} / {n_avail}'
        else:
            selected = image_paths
            note = f'all {n_avail}'

        target_dir = ws / spec['style_id']
        target_dir.mkdir(parents=True, exist_ok=True)

        meta_rows = []
        link_mode = 'symlink'
        for idx, src in enumerate(selected):
            dst_name = f'{idx:05d}_{src.name}'
            dst = target_dir / dst_name
            if link_mode == 'symlink':
                try:
                    os.symlink(src, dst)
                except OSError:
                    link_mode = 'copy'
                    print(f'[{spec["style_id"]}] Symlinks unavailable → copying files (uses disk space).')
                    shutil.copy2(src, dst)
                    total_copy_bytes += src.stat().st_size
            else:
                shutil.copy2(src, dst)
                total_copy_bytes += src.stat().st_size
            meta_rows.append({'file_name': dst_name, 'text': spec['caption']})

        with (target_dir / 'metadata.jsonl').open('w', encoding='utf-8') as fh:
            for row in meta_rows:
                fh.write(json.dumps(row, ensure_ascii=False) + '\n')

        total_images += len(selected)
        prepared.append({
            'style_id':         spec['style_id'],
            'display_name':     spec['display_name'],
            'dataset_dir':      str(target_dir),
            'available_images': n_avail,
            'selected_images':  len(selected),
            'selection_note':   note,
            'link_mode':        link_mode,
        })

    df = pd.DataFrame(prepared)
    display(df)
    print(f'Total workspace images: {total_images}')
    if total_copy_bytes:
        print(f'WARNING: {total_copy_bytes / 1024**3:.2f} GB used by file copies.')
    else:
        print('All files linked via symlinks — minimal disk usage.')
    return df

prepared_datasets = prepare_style_workspace(CONFIG, STYLE_SPECS)

## Step 6 — Training Script Setup

Downloads a pinned version of the Diffusers LoRA training script.  
Falls back to `main` branch if the release tag is not available.  
All fixes (including `--report_to=tensorboard`) are applied **in-place** to the downloaded script so the notebook works even when the upstream script changes.

In [ ]:
import importlib.metadata as _meta
import os
import sys
import urllib.request
from pathlib import Path

# ── Prerequisites check ─────────────────────────────────────────────────────
def check_prerequisites():
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f'CUDA available: {cuda_ok}')
    if cuda_ok:
        for i in range(torch.cuda.device_count()):
            free, total = [x / 1024**3 for x in torch.cuda.mem_get_info(i)]
            print(f'  GPU {i}: {torch.cuda.get_device_name(i)} — {free:.1f}/{total:.1f} GB free')

    try:
        import bitsandbytes as bnb
        print(f'bitsandbytes: {bnb.__version__} ✓')
    except ImportError:
        print('bitsandbytes: not available — use_8bit_adam will be disabled.')

    import accelerate
    print(f'accelerate: {accelerate.__version__} ✓')
    import diffusers
    print(f'diffusers: {diffusers.__version__} ✓')

check_prerequisites()

# ── Detect whether bitsandbytes is operational ───────────────────────────────
def _bnb_available() -> bool:
    try:
        import bitsandbytes as bnb
        import torch
        # Quick functional check
        dummy = torch.zeros(4, 4, device='cuda' if torch.cuda.is_available() else 'cpu')
        _ = bnb.optim.Adam8bit([dummy], lr=1e-4)
        return True
    except Exception as exc:
        print(f'bitsandbytes check failed ({exc}) — falling back to AdamW')
        return False

BNB_OK = CONFIG.use_8bit_adam and _bnb_available()
print(f'8-bit Adam: {"enabled" if BNB_OK else "disabled (using AdamW)"}')

# ── Download training script ─────────────────────────────────────────────────
# We try a pinned SD1.5 LoRA script that is known-good, then fall back to main.
SCRIPT_URLS = [
    # Pinned to a tag where 'none' was still supported — but we patch it below anyway
    'https://raw.githubusercontent.com/huggingface/diffusers/v0.27.2/examples/text_to_image/train_text_to_image_lora.py',
    # Fallback to the latest diffusers release tag
    f'https://raw.githubusercontent.com/huggingface/diffusers/v{_meta.version("diffusers")}/examples/text_to_image/train_text_to_image_lora.py',
    # Last-resort: main branch
    'https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora.py',
]

def ensure_training_script(config) -> Path:
    config.temp_root.mkdir(parents=True, exist_ok=True)
    script_path = config.temp_root / 'train_text_to_image_lora.py'

    if script_path.exists() and script_path.stat().st_size > 5_000:
        print(f'Training script already cached: {script_path}')
        return script_path

    for url in SCRIPT_URLS:
        try:
            print(f'Downloading: {url}')
            urllib.request.urlretrieve(url, script_path)
            if script_path.stat().st_size > 5_000:
                print(f'Downloaded {script_path.stat().st_size // 1024} KB')
                return script_path
            script_path.unlink(missing_ok=True)
        except Exception as exc:
            print(f'  → failed: {exc}')

    raise RuntimeError(
        'Could not download training script. '
        'Enable internet access in Kaggle notebook settings.'
    )

training_script_path = ensure_training_script(CONFIG)

# ── Patch the script (robust, idempotent) ────────────────────────────────────
# Problem 1: accelerate >=0.31 removed LoggerType.NONE
#   Fix: replace 'report_to="none"' with 'report_to="tensorboard"' in the script
# Problem 2: some script versions hardcode 'none' as default in argparse
#   Fix: replace the argparse default as well
print('Applying compatibility patches to training script...')
script_src = training_script_path.read_text(encoding='utf-8')
patches = [
    # argparse default 'none' → 'tensorboard'
    ('default="none"',       'default="tensorboard"'),
    ("default='none'",       "default='tensorboard'"),
    # any --report_to none that might be hardcoded
    ('report_to="none"',     'report_to="tensorboard"'),
    ("report_to='none'",     "report_to='tensorboard'"),
    # LoggerType.NONE reference
    ('LoggerType.NONE',      'LoggerType.TENSORBOARD'),
]
for old, new in patches:
    script_src = script_src.replace(old, new)
training_script_path.write_text(script_src, encoding='utf-8')
print('Patches applied.')

# ── Command builder ──────────────────────────────────────────────────────────
def build_train_command(config, dataset_dir, output_dir, training_script, bnb_ok):
    cmd = [
        sys.executable,
        str(training_script),
        f'--pretrained_model_name_or_path={config.base_model}',
        f'--train_data_dir={dataset_dir}',
        '--caption_column=text',
        f'--resolution={config.resolution}',
        '--random_flip',
        f'--train_batch_size={config.train_batch_size}',
        f'--gradient_accumulation_steps={config.gradient_accumulation_steps}',
        f'--max_train_steps={config.max_train_steps}',
        f'--learning_rate={config.learning_rate}',
        '--lr_scheduler=cosine',
        '--lr_warmup_steps=50',
        f'--seed={config.seed}',
        f'--output_dir={output_dir}',
        f'--rank={config.rank}',
        '--gradient_checkpointing',
        '--mixed_precision=fp16',
        '--allow_tf32',
        '--report_to=tensorboard',
        '--checkpointing_steps=9999',    # skip intermediate checkpoints to save disk
    ]
    if bnb_ok:
        cmd.append('--use_8bit_adam')
    return cmd

print(f'Training script ready: {training_script_path}')

## Step 7 — Training Execution

| Mode | Behaviour |
|---|---|
| `single_gpu` | Trains all styles sequentially on GPU 0 |
| `dual_gpu_parallel` | Trains two styles in parallel (requires 2×T4) |

Expected wall-clock time on a single T4: ~45 min per style → ~3h total.

In [ ]:
import gc
import glob
import os
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd
import torch

# ── Runtime environment ──────────────────────────────────────────────────────
def build_env(config, gpu_id):
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES']      = str(gpu_id)
    env['HF_HOME']                   = str(config.temp_root / 'hf-home')
    env['HF_DATASETS_CACHE']         = str(config.temp_root / 'hf-datasets')
    env['TRANSFORMERS_CACHE']        = str(config.temp_root / 'hf-transformers')
    env['HUGGINGFACE_HUB_CACHE']     = str(config.temp_root / 'hf-hub')
    env['TMPDIR']                    = str(config.temp_root / 'tmp')
    env['TOKENIZERS_PARALLELISM']    = 'false'
    env['PYTORCH_CUDA_ALLOC_CONF']   = 'expandable_segments:True'
    env['MPLBACKEND']                = 'Agg'
    env['ACCELERATE_LOG_LEVEL']      = 'WARNING'
    # Suppress cuDNN / cuBLAS duplicate-registration warnings
    env['TF_CPP_MIN_LOG_LEVEL']      = '3'
    # Ensure temp subdir exists
    (config.temp_root / 'tmp').mkdir(parents=True, exist_ok=True)
    return env

# ── Log helper ───────────────────────────────────────────────────────────────
def print_log_tail(log_path, tail_lines=30):
    if not log_path.exists():
        print(f'Log not found: {log_path}')
        return
    lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
    tail = lines[-tail_lines:]
    print(f'\n--- Last {len(tail)} lines of {log_path.name} ---')
    for l in tail:
        print(l)
    print('--- End of log ---')

# ── Weights finder (robust against naming variations) ────────────────────────
def find_lora_weights(output_dir: Path) -> Path:
    """Search for the LoRA weights file regardless of its exact name."""
    candidates = [
        output_dir / 'pytorch_lora_weights.safetensors',
        output_dir / 'adapter_model.safetensors',
    ]
    for c in candidates:
        if c.exists():
            return c
    # Wildcard fallback
    safetensors = sorted(output_dir.glob('*.safetensors'))
    if safetensors:
        return safetensors[0]
    bin_files = sorted(output_dir.glob('*.bin'))
    if bin_files:
        return bin_files[0]
    return None

# ── Launch one training job ───────────────────────────────────────────────────
def launch_style_training(config, prepared_row, training_script, gpu_id):
    weights_root = config.output_root / 'weights'
    logs_root    = config.output_root / 'logs'
    weights_root.mkdir(parents=True, exist_ok=True)
    logs_root.mkdir(parents=True, exist_ok=True)

    style_id   = prepared_row['style_id']
    output_dir = weights_root / f'lora-output-{style_id}'
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    log_path = logs_root / f'{style_id}.log'
    command  = build_train_command(
        config, prepared_row['dataset_dir'], output_dir, training_script, BNB_OK
    )

    print(f'[{style_id}] Launching training on GPU {gpu_id}...')
    print(f'[{style_id}] Steps: {config.max_train_steps} | Rank: {config.rank} | Images: {prepared_row["selected_images"]}')

    log_handle = open(log_path, 'w', encoding='utf-8')
    process    = subprocess.Popen(
        command,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=build_env(config, gpu_id),
    )
    return process, log_handle, log_path, output_dir

# ── Collect results ──────────────────────────────────────────────────────────
def wait_and_collect(process, log_handle, log_path, output_dir, style_id):
    return_code = process.wait()
    log_handle.close()

    if return_code != 0:
        print(f'[{style_id}] ❌ FAILED with exit code {return_code}')
        print_log_tail(log_path, tail_lines=40)
        raise RuntimeError(f'Training failed for {style_id} (exit code {return_code}).')

    weight_path = find_lora_weights(output_dir)
    if weight_path is None:
        print(f'[{style_id}] ⚠️  Exit 0 but no weights found in {output_dir}')
        print_log_tail(log_path)
        raise RuntimeError(f'No weights found for {style_id}.')

    # Normalise to expected name
    canonical = output_dir / 'pytorch_lora_weights.safetensors'
    if weight_path != canonical:
        shutil.copy2(weight_path, canonical)
        print(f'[{style_id}] Copied {weight_path.name} → pytorch_lora_weights.safetensors')
        weight_path = canonical

    size_mb = weight_path.stat().st_size / 1024 ** 2
    print(f'[{style_id}] ✅ SUCCESS  →  {weight_path}  ({size_mb:.1f} MB)')
    return {
        'style_id':      style_id,
        'output_dir':    str(output_dir),
        'weights_path':  str(weight_path),
        'log_path':      str(log_path),
        'weight_size_mb': size_mb,
    }

# ── Execution modes ───────────────────────────────────────────────────────────
def run_single_gpu(config, prepared_frame, training_script):
    results = []
    for row in prepared_frame.to_dict(orient='records'):
        t0 = time.time()
        process, log_handle, log_path, output_dir = launch_style_training(
            config, row, training_script, gpu_id=0
        )
        result = wait_and_collect(process, log_handle, log_path, output_dir, row['style_id'])
        elapsed = (time.time() - t0) / 60
        print(f'  [{row["style_id"]}] Wall time: {elapsed:.1f} min')
        results.append(result)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pd.DataFrame(results)

def run_dual_gpu_parallel(config, prepared_frame, training_script):
    if torch.cuda.device_count() < 2:
        raise RuntimeError('dual_gpu_parallel mode requires at least 2 visible GPUs.')

    pending   = prepared_frame.to_dict(orient='records')
    free_gpus = [0, 1]
    active    = []
    completed = []

    while pending or active:
        while pending and free_gpus:
            gpu_id = free_gpus.pop(0)
            row    = pending.pop(0)
            proc, lh, lp, od = launch_style_training(config, row, training_script, gpu_id)
            active.append(dict(style_id=row['style_id'], gpu_id=gpu_id,
                               process=proc, log_handle=lh, log_path=lp, output_dir=od,
                               t0=time.time()))

        time.sleep(15)
        still_running = []
        for job in active:
            rc = job['process'].poll()
            if rc is None:
                still_running.append(job)
                continue
            result = wait_and_collect(
                job['process'], job['log_handle'],
                job['log_path'], job['output_dir'], job['style_id'],
            )
            elapsed = (time.time() - job['t0']) / 60
            print(f'  [{job["style_id"]}] Wall time: {elapsed:.1f} min')
            free_gpus.append(job['gpu_id'])
            free_gpus.sort()
            completed.append(result)
        active = still_running

    return pd.DataFrame(completed).sort_values('style_id').reset_index(drop=True)

def run_training(config, prepared_frame, training_script):
    config.output_root.mkdir(parents=True, exist_ok=True)
    mode = config.train_mode
    if mode == 'single_gpu':
        return run_single_gpu(config, prepared_frame, training_script)
    if mode == 'dual_gpu_parallel':
        return run_dual_gpu_parallel(config, prepared_frame, training_script)
    raise ValueError(f'Unsupported training mode: {mode!r}. Use "single_gpu" or "dual_gpu_parallel".')

# ── Run ───────────────────────────────────────────────────────────────────────
weights_manifest = run_training(CONFIG, prepared_datasets, training_script_path)
display(weights_manifest)

weights_manifest_path = CONFIG.output_root / 'reports' / 'weights_manifest.json'
weights_manifest_path.parent.mkdir(parents=True, exist_ok=True)
weights_manifest_path.write_text(
    weights_manifest.to_json(orient='records', indent=2), encoding='utf-8'
)
print(f'Manifest saved: {weights_manifest_path}')

## Step 8 — Style Transfer Showcase

Loads real-life content photos and applies each trained LoRA in img2img mode to produce a before/after comparison grid.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image

content_dir = CONFIG.content_images_path
content_images = sorted(
    p for p in content_dir.rglob('*')
    if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
)
print(f'Found {len(content_images)} content images')

NUM_CONTENT = min(4, len(content_images))
selected_content = content_images[:NUM_CONTENT]

figures_root = CONFIG.output_root / 'figures'
figures_root.mkdir(parents=True, exist_ok=True)
grid_path = figures_root / 'artoria_style_transfer_showcase.png'

# Load base pipeline (no weights)
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    CONFIG.base_model,
    torch_dtype=torch.float16,
    safety_checker=None,
).to('cuda')
pipe.set_progress_bar_config(disable=True)

num_cols = 1 + len(STYLE_SPECS)
fig, axes = plt.subplots(NUM_CONTENT, num_cols, figsize=(5 * num_cols, 5 * NUM_CONTENT))
if NUM_CONTENT == 1:
    axes = axes.reshape(1, -1)

col_labels = ['Original'] + [s['display_name'] for s in STYLE_SPECS]
for col, label in enumerate(col_labels):
    axes[0][col].set_title(label, fontsize=13, fontweight='bold')

for row_idx, content_path in enumerate(selected_content):
    init_image = Image.open(content_path).convert('RGB').resize((512, 512))
    axes[row_idx][0].imshow(init_image)
    axes[row_idx][0].axis('off')

    for style_idx, spec in enumerate(STYLE_SPECS):
        col        = style_idx + 1
        weight_dir = CONFIG.output_root / 'weights' / f'lora-output-{spec["style_id"]}'

        # Find weights regardless of file name
        weight_file = find_lora_weights(weight_dir)
        if weight_file is None:
            print(f'  No weights for {spec["style_id"]} — skipping')
            axes[row_idx][col].axis('off')
            continue

        try:
            pipe.unload_lora_weights()
        except Exception:
            pass

        # load_lora_weights accepts a directory OR a file path
        try:
            pipe.load_lora_weights(str(weight_dir))
        except Exception:
            pipe.load_lora_weights(str(weight_file))

        styled = pipe(
            prompt=spec['prompt_suffix'],
            image=init_image,
            strength=0.75,
            num_inference_steps=30,
            guidance_scale=7.5,
            generator=torch.Generator(device='cuda').manual_seed(CONFIG.seed + row_idx * 10 + style_idx),
        ).images[0]

        axes[row_idx][col].imshow(styled)
        axes[row_idx][col].axis('off')

fig.suptitle('Artoria Neural Style Transfer — LoRA Results', fontsize=16, fontweight='bold', y=1.01)
fig.tight_layout()
fig.savefig(grid_path, dpi=150, bbox_inches='tight')
plt.show()

try:
    pipe.unload_lora_weights()
except Exception:
    pass
del pipe
torch.cuda.empty_cache()
print(f'Grid saved: {grid_path}')

## Step 9 — Package Release Artifacts

In [ ]:
import json
import os
import shutil
import subprocess
from dataclasses import asdict
from pathlib import Path

from IPython.display import FileLink, display

reports_root = CONFIG.output_root / 'reports'
reports_root.mkdir(parents=True, exist_ok=True)
summary_path = reports_root / 'run_summary.json'
release_base = Path('/kaggle/working/artoria_style_transfer_release')
release_name = 'artoria_style_transfer_release.zip'
release_zip_path = Path('/kaggle/working') / release_name

run_summary = {
    'project_name':          CONFIG.project_name,
    'course':                'Machine Learning Fundamentals',
    'department':            'Artificial Intelligence and Data Engineering',
    'dataset_origin':        'Team-created Artoria dataset',
    'selected_topic':        'Neural style transfer with LoRA adapters',
    'base_model':            CONFIG.base_model,
    'training_config':       {
        k: str(v) if isinstance(v, Path) else v
        for k, v in asdict(CONFIG).items()
    },
    'styles':                STYLE_SPECS,
    'weights_manifest':      json.loads(weights_manifest.to_json(orient='records')),
    'validation_grid':       str(CONFIG.output_root / 'figures' / 'artoria_style_transfer_showcase.png'),
}

summary_path.write_text(json.dumps(run_summary, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Run summary: {summary_path}')

if CONFIG.workspace_root.exists():
    shutil.rmtree(CONFIG.workspace_root)

if release_zip_path.exists():
    release_zip_path.unlink()

subprocess.run(['zip', '-r', '-q', str(release_zip_path), '.'], cwd=str(CONFIG.output_root))

zip_size_mb = release_zip_path.stat().st_size / 1024 ** 2
print(f'Release zip: {release_zip_path}  ({zip_size_mb:.1f} MB)')

os.chdir('/kaggle/working')
display(FileLink(release_name))
